In [ ]:
"""
LLPS PEPTIDE CLASSIFICATION - BIOINFORMATIC ANALYSIS WITH ROC CURVES AND MODEL COMPARISON
==============================================
Features:
  1. Position-specific amino acid encoding
  2. Amino acid composition & charge properties
  3. Hydrophobic moment & amphipathicity
  4. Aromatic geometry
  5. Lipidation features
  6. LOW-COMPLEXITY REGION DETECTION
  7. SECONDARY STRUCTURE PROPENSITIES
  8. MULTIPLE MODELS COMPARISON
  9. ROC CURVES (INTERNAL & EXTERNAL VALIDATION)

Workflow: Training → Feature Extraction → Multi-Model Training → ROC Analysis → Validation
"""

# ============================================================================
# CONFIGURATION
# ============================================================================

CONFIG = {
    # Paths
    #Replace with actual file path location
    'output_dir': "data/output",
    'training_file': "data/input.xlsx",
    'experimental_file': "data/38-experimental-validation.xlsx",
    
    # Training
    'random_state': 42,
    'test_size': 0.2,
    
    # Feature selection
    'use_complexity_features': True,
    'use_ss_features': True,
}

# ============================================================================
# IMPORTS
# ============================================================================

import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Tuple, List, Dict
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    roc_auc_score, f1_score, matthews_corrcoef, accuracy_score,
    balanced_accuracy_score, confusion_matrix, precision_score, recall_score,
    roc_curve, auc, classification_report
)

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from lightgbm import LGBMClassifier

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("[WARNING] XGBoost not installed")

from imblearn.over_sampling import SMOTE
import joblib

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# ============================================================================
# PART 1: BIOINFORMATIC FEATURE EXTRACTORS
# ============================================================================

AA_ALPHABET = ['W', 'R', 'N', 'D', 'Y', 'K', 'F']
N_AA = len(AA_ALPHABET)


def encode_positional(sequences, max_len=8):
    """Position-specific one-hot encoding."""
    encoded = []
    for seq in sequences:
        vec = []
        for pos in range(max_len):
            if pos < len(seq):
                aa = seq[pos]
                one_hot = [1 if aa == a else 0 for a in AA_ALPHABET]
            else:
                one_hot = [0] * N_AA
            vec.extend(one_hot)
        encoded.append(vec)
    feature_names = [f'pos{p}_{aa}' for p in range(max_len) for aa in AA_ALPHABET]
    return np.array(encoded, dtype=np.float64), feature_names


def aa_composition_features(sequences):
    """Amino acid composition and properties."""
    rows = []
    for seq in sequences:
        n = len(seq)
        row = {}
        for aa in AA_ALPHABET:
            row[f'frac_{aa}'] = seq.count(aa) / n
        row['frac_aromatic'] = sum(seq.count(aa) for aa in 'WYF') / n
        row['frac_positive'] = sum(seq.count(aa) for aa in 'KR') / n
        row['frac_negative'] = sum(seq.count(aa) for aa in 'D') / n
        rows.append(row)
    return pd.DataFrame(rows)


def charge_features(sequences):
    """Charge distribution and properties."""
    charge_map = {'K': 1, 'R': 1, 'D': -1, 'N': 0, 'W': 0, 'Y': 0, 'F': 0}
    rows = []
    for seq in sequences:
        n = len(seq)
        charges = [charge_map.get(aa, 0) for aa in seq]
        net_charge = sum(charges)
        abs_charge = sum(abs(c) for c in charges)
        
        pos_positions = [i for i, c in enumerate(charges) if c > 0]
        neg_positions = [i for i, c in enumerate(charges) if c < 0]
        
        if pos_positions and neg_positions:
            charge_separation = np.mean([abs(p - ng) for p in pos_positions for ng in neg_positions])
            max_charge_sep = max(abs(p - ng) for p in pos_positions for ng in neg_positions)
        else:
            charge_separation = 0
            max_charge_sep = 0
        
        rows.append({
            'net_charge': net_charge,
            'abs_charge': abs_charge,
            'charge_density': abs_charge / n,
            'charge_separation': charge_separation,
            'max_charge_separation': max_charge_sep,
            'n_positive': len(pos_positions),
            'n_negative': len(neg_positions),
        })
    return pd.DataFrame(rows)


def hydrophobic_moment_features(sequences):
    """Hydrophobic moment and amphipathicity."""
    kd_hydro = {'W': 1.37, 'Y': 0.49, 'F': 1.13, 'K': -1.50, 'R': -0.99, 'D': -3.27, 'N': -0.90}
    rows = []
    for seq in sequences:
        n = len(seq)
        hv = [kd_hydro.get(aa, 0) for aa in seq]
        mx = sum(hv[i] * np.cos(2 * np.pi * i / n) for i in range(n))
        my = sum(hv[i] * np.sin(2 * np.pi * i / n) for i in range(n))
        hydro_moment = np.sqrt(mx ** 2 + my ** 2) / n
        
        n_hpho = sum(1 for v in hv if v > 0.3)
        n_hphi = sum(1 for v in hv if v < -0.5)
        amphipathicity = (n_hpho * n_hphi) / (n * n)
        
        rows.append({
            'hydro_moment': hydro_moment,
            'mean_hydropathy': np.mean(hv),
            'std_hydropathy': np.std(hv),
            'amphipathicity': amphipathicity,
            'n_hydrophobic': n_hpho,
            'n_hydrophilic': n_hphi,
        })
    return pd.DataFrame(rows)


def aromatic_geometry_features(sequences):
    """Aromatic residue spatial arrangement."""
    rows = []
    for seq in sequences:
        ap = [i for i, aa in enumerate(seq) if aa in 'WYF']
        na = len(ap)
        
        if na < 2:
            rows.append({
                'n_aromatic': na,
                'aromatic_fraction': na / len(seq),
                'aromatic_contact_potential': 0,
                'stacking_favorable_pairs': 0,
                'pi_pi_distance_min': 0,
            })
            continue
        
        dists = []
        for ii in range(len(ap)):
            for jj in range(ii + 1, len(ap)):
                dists.append(abs(ap[jj] - ap[ii]))
        
        fav = sum(1 for d in dists if 2 <= d <= 4)
        rows.append({
            'n_aromatic': na,
            'aromatic_fraction': na / len(seq),
            'aromatic_contact_potential': fav / len(dists) if dists else 0,
            'stacking_favorable_pairs': fav,
            'pi_pi_distance_min': min(dists) if dists else 0,
        })
    return pd.DataFrame(rows)


def lipidation_features(sequences, lipidation_values):
    """Lipidation properties."""
    lp = {
        'octanoyl': {'has': 1, 'logp': 2.8},
        'hexanoyl': {'has': 1, 'logp': 2.0},
        'yes': {'has': 1, 'logp': 2.4},
        'no': {'has': 0, 'logp': 0.0},
        'none': {'has': 0, 'logp': 0.0},
    }
    rows = []
    for seq, lip in zip(sequences, lipidation_values):
        ll = str(lip).strip().lower()
        p = lp.get(ll, lp['no'])
        rows.append({
            'has_lipidation': p['has'],
            'lipid_logp': p['logp'],
        })
    return pd.DataFrame(rows)


def low_complexity_features(sequences):
    """Detect low-complexity regions (repetitive or homopolymer-like)."""
    rows = []
    for seq in sequences:
        n = len(seq)
        window_size = 5
        
        aa_counts = {}
        for aa in seq:
            aa_counts[aa] = aa_counts.get(aa, 0) + 1
        
        total = sum(aa_counts.values())
        full_entropy = -sum((count/total) * np.log2(count/total + 1e-10) 
                           for count in aa_counts.values())
        
        max_entropy = np.log2(20)
        normalized_entropy = full_entropy / max_entropy
        
        window_entropies = []
        for i in range(len(seq) - window_size + 1):
            window = seq[i:i+window_size]
            w_counts = {}
            for aa in window:
                w_counts[aa] = w_counts.get(aa, 0) + 1
            
            w_entropy = -sum((cnt/window_size) * np.log2(cnt/window_size + 1e-10) 
                            for cnt in w_counts.values())
            window_entropies.append(w_entropy)
        
        mean_window_entropy = np.mean(window_entropies) if window_entropies else 0
        min_window_entropy = np.min(window_entropies) if window_entropies else 0
        
        low_complexity_windows = sum(1 for e in window_entropies if e < 1.0)
        frac_low_complexity = low_complexity_windows / len(window_entropies) if window_entropies else 0
        
        max_run_length = 1
        current_run = 1
        for i in range(1, len(seq)):
            if seq[i] == seq[i-1]:
                current_run += 1
                max_run_length = max(max_run_length, current_run)
            else:
                current_run = 1
        
        dipeptide_repeats = 0
        for i in range(0, len(seq) - 3, 2):
            if seq[i:i+2] == seq[i+2:i+4]:
                dipeptide_repeats += 1
        
        rows.append({
            'complexity_full_entropy': normalized_entropy,
            'complexity_window_mean': mean_window_entropy,
            'complexity_window_min': min_window_entropy,
            'complexity_frac_low': frac_low_complexity,
            'complexity_max_run': max_run_length,
            'complexity_dipeptide_repeats': dipeptide_repeats,
        })
    
    return pd.DataFrame(rows)


def secondary_structure_features(sequences):
    """Calculate secondary structure propensities using Chou-Fasman values."""
    
    cf_helix = {
        'A': 1.42, 'R': 0.98, 'N': 0.67, 'D': 1.01, 'C': 0.70,
        'E': 1.51, 'Q': 1.11, 'G': 0.57, 'H': 1.00, 'I': 1.08,
        'L': 1.21, 'K': 1.16, 'M': 1.45, 'F': 1.13, 'P': 0.57,
        'S': 0.77, 'T': 0.83, 'W': 1.08, 'Y': 0.69, 'V': 1.06,
    }
    
    cf_sheet = {
        'A': 0.83, 'R': 0.93, 'N': 0.89, 'D': 0.54, 'C': 1.19,
        'E': 0.37, 'Q': 1.10, 'G': 0.75, 'H': 0.87, 'I': 1.60,
        'L': 1.30, 'K': 0.74, 'M': 1.05, 'F': 1.38, 'P': 0.55,
        'S': 0.75, 'T': 1.19, 'W': 1.37, 'Y': 1.47, 'V': 1.70,
    }
    
    cf_turn = {
        'A': 0.66, 'R': 0.95, 'N': 1.56, 'D': 1.46, 'C': 1.19,
        'E': 0.74, 'Q': 1.11, 'G': 1.57, 'H': 0.87, 'I': 0.60,
        'L': 0.57, 'K': 1.01, 'M': 0.60, 'F': 0.60, 'P': 1.52,
        'S': 1.42, 'T': 0.83, 'W': 0.96, 'Y': 1.14, 'V': 0.50,
    }
    
    rows = []
    for seq in sequences:
        n = len(seq)
        
        helix_scores = [cf_helix.get(aa, 0.8) for aa in seq]
        sheet_scores = [cf_sheet.get(aa, 0.8) for aa in seq]
        turn_scores = [cf_turn.get(aa, 0.8) for aa in seq]
        
        mean_helix_prop = np.mean(helix_scores)
        mean_sheet_prop = np.mean(sheet_scores)
        mean_turn_prop = np.mean(turn_scores)
        
        helix_favoring = sum(1 for h, s in zip(helix_scores, sheet_scores) if h > s) / n
        sheet_favoring = sum(1 for s, h in zip(sheet_scores, helix_scores) if s > h) / n
        
        n_strong_helix = sum(1 for h in helix_scores if h > 1.2)
        n_strong_sheet = sum(1 for s in sheet_scores if s > 1.4)
        
        helix_strength = np.std(helix_scores)
        sheet_strength = np.std(sheet_scores)
        
        rows.append({
            'ss_helix_propensity': mean_helix_prop,
            'ss_sheet_propensity': mean_sheet_prop,
            'ss_turn_propensity': mean_turn_prop,
            'ss_helix_favoring': helix_favoring,
            'ss_sheet_favoring': sheet_favoring,
            'ss_strong_helix_residues': n_strong_helix / n,
            'ss_strong_sheet_residues': n_strong_sheet / n,
            'ss_helix_strength': helix_strength,
            'ss_sheet_strength': sheet_strength,
        })
    
    return pd.DataFrame(rows)


# ============================================================================
# PART 2: BIOINFORMATIC FEATURE EXTRACTION ENGINE
# ============================================================================

class BioinformaticFeatureExtractor:
    """Extract comprehensive bioinformatic properties."""
    
    def __init__(self, config: Dict):
        self.max_len = 8
        self.config = config
        
    def extract_all_features(self, sequences: np.ndarray, 
                            lipidations: np.ndarray) -> Tuple[np.ndarray, List[str]]:
        """Extract all bioinformatic features."""
        print("\n[BIOINFORMATICS] Computing comprehensive bioinformatic features...")
        
        feature_dfs = []
        feature_names_list = []
        
        print("  → Position encoding...")
        X_pos, names_pos = encode_positional(sequences, self.max_len)
        feature_dfs.append(X_pos)
        feature_names_list.append(names_pos)
        
        print("  → Amino acid composition...")
        df_comp = aa_composition_features(sequences)
        feature_dfs.append(df_comp.values)
        feature_names_list.append(df_comp.columns.tolist())
        
        print("  → Charge properties...")
        df_charge = charge_features(sequences)
        feature_dfs.append(df_charge.values)
        feature_names_list.append(df_charge.columns.tolist())
        
        print("  → Hydrophobic moment...")
        df_hydro = hydrophobic_moment_features(sequences)
        feature_dfs.append(df_hydro.values)
        feature_names_list.append(df_hydro.columns.tolist())
        
        print("  → Aromatic geometry...")
        df_arom = aromatic_geometry_features(sequences)
        feature_dfs.append(df_arom.values)
        feature_names_list.append(df_arom.columns.tolist())
        
        print("  → Lipidation properties...")
        df_lipid = lipidation_features(sequences, lipidations)
        feature_dfs.append(df_lipid.values)
        feature_names_list.append(df_lipid.columns.tolist())
        
        if self.config.get('use_complexity_features', True):
            print("  → Low-complexity region detection...")
            df_complexity = low_complexity_features(sequences)
            feature_dfs.append(df_complexity.values)
            feature_names_list.append(df_complexity.columns.tolist())
        
        if self.config.get('use_ss_features', True):
            print("  → Secondary structure propensities (Chou-Fasman)...")
            df_ss = secondary_structure_features(sequences)
            feature_dfs.append(df_ss.values)
            feature_names_list.append(df_ss.columns.tolist())
        
        X_bio = np.hstack(feature_dfs)
        feature_names = [name for names in feature_names_list for name in names]
        
        print(f"\n  ✓ Bioinformatic features shape: {X_bio.shape}")
        
        return X_bio, feature_names


# ============================================================================
# PART 3: ROC CURVE VISUALIZATION ENGINE
# ============================================================================

class ROCVisualization:
    """Visualize ROC curves for multi-class classification."""
    
    def __init__(self, colors=None):
        self.colors = colors or sns.color_palette("husl", 10)
    
    def plot_macro_roc_curve(self, y_true, y_pred_proba, class_names, 
                            title, output_path, dpi=300):
        """Plot macro-averaged ROC curve for multi-class classification."""
        
        n_classes = len(class_names)
        y_true_bin = label_binarize(y_true, classes=list(range(n_classes)))
        
        fpr_list = []
        tpr_list = []
        auc_per_class = {}
        
        for i in range(n_classes):
            if y_true_bin[:, i].sum() == 0:
                print(f"    ⚠️  No positive samples for class {class_names[i]}")
                continue
            
            fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
            roc_auc = auc(fpr, tpr)
            auc_per_class[class_names[i]] = roc_auc
            fpr_list.append(fpr)
            tpr_list.append(tpr)
        
        mean_fpr = np.linspace(0, 1, 100)
        tpr_interp_list = []
        
        for fpr, tpr in zip(fpr_list, tpr_list):
            tpr_interp = np.interp(mean_fpr, fpr, tpr)
            tpr_interp_list.append(tpr_interp)
        
        mean_tpr = np.mean(tpr_interp_list, axis=0)
        mean_tpr[0], mean_tpr[-1] = 0, 1
        macro_auc = auc(mean_fpr, mean_tpr)
        
        fig, ax = plt.subplots(figsize=(8, 6), dpi=dpi)
        
        ax.plot(mean_fpr, mean_tpr, 
                label=f'Macro-avg ROC (AUC = {macro_auc:.3f})', 
                lw=3.5, color='darkblue')
        ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
        
        ax.set_xlim([0.0, 1.0])
        ax.set_ylim([0.0, 1.05])
        ax.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
        ax.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.legend(loc='lower right', fontsize=11)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(output_path, dpi=dpi, bbox_inches='tight')
        plt.close()
        
        roc_csv_filename = output_path.stem.replace('roc_', 'roc_data_') + '.csv'
        roc_csv_path = output_path.parent / roc_csv_filename
        roc_df = pd.DataFrame({
            'FPR': mean_fpr,
            'TPR': mean_tpr,
            'Macro_AUC': macro_auc
        })
        roc_df.to_csv(roc_csv_path, index=False)
        
        print(f"    ✓ Macro ROC curve: {output_path.name}")
        print(f"    ✓ ROC data saved: {roc_csv_path.name}")
        
        return mean_fpr, mean_tpr, macro_auc, auc_per_class
    
    def plot_per_class_roc_curves(self, y_true, y_pred_proba, class_names,
                                 title, output_path, dpi=300):
        """Plot per-class (One-vs-Rest) ROC curves."""
        
        n_classes = len(class_names)
        y_true_bin = label_binarize(y_true, classes=list(range(n_classes)))
        
        fig, axes = plt.subplots(1, n_classes, figsize=(6*n_classes, 5.5), dpi=dpi)
        if n_classes == 1:
            axes = [axes]
        
        colors_palette = sns.color_palette("husl", n_classes)
        
        for i, class_name in enumerate(class_names):
            ax = axes[i]
            
            if y_true_bin[:, i].sum() == 0:
                ax.text(0.5, 0.5, f'No positive samples\nfor {class_name}',
                       ha='center', va='center', fontsize=12, transform=ax.transAxes)
                continue
            
            fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
            roc_auc = auc(fpr, tpr)
            
            ax.plot(fpr, tpr, color=colors_palette[i], linewidth=3,
                   label=f'{class_name}\n(AUC = {roc_auc:.3f})')
            ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Chance')
            
            ax.set_xlim([0.0, 1.0])
            ax.set_ylim([0.0, 1.05])
            ax.set_xlabel('False Positive Rate', fontsize=11, fontweight='bold')
            ax.set_ylabel('True Positive Rate', fontsize=11, fontweight='bold')
            ax.set_title(f'ROC: {class_name} (One-vs-Rest)', fontsize=12, fontweight='bold')
            ax.legend(loc='lower right', fontsize=10)
            ax.grid(True, alpha=0.3)
        
        plt.suptitle(title, fontsize=14, fontweight='bold', y=1.00)
        plt.tight_layout()
        plt.savefig(output_path, dpi=dpi, bbox_inches='tight')
        plt.close()
        
        print(f"    ✓ Per-class ROC curves: {output_path.name}")
    
    def plot_roc_comparison(self, datasets_dict, output_path, dpi=300):
        """Compare macro ROC curves across multiple datasets."""
        
        fig, ax = plt.subplots(figsize=(10, 7), dpi=dpi)
        
        colors_datasets = {'Internal Test': 'darkblue', 
                          'External Validation': 'darkred'}
        
        for dataset_name, (fpr, tpr, auc_val) in datasets_dict.items():
            color = colors_datasets.get(dataset_name, 'steelblue')
            ax.plot(fpr, tpr, label=f'{dataset_name} (AUC = {auc_val:.3f})',
                   lw=3, color=color)
        
        ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
        
        ax.set_xlim([0.0, 1.0])
        ax.set_ylim([0.0, 1.05])
        ax.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
        ax.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
        ax.set_title('ROC Curve Comparison: Internal vs External',
                    fontsize=14, fontweight='bold')
        ax.legend(loc='lower right', fontsize=11)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(output_path, dpi=dpi, bbox_inches='tight')
        plt.close()
        
        print(f"    ✓ ROC comparison: {output_path.name}")


# ============================================================================
# PART 4: MULTI-MODEL TRAINING ENGINE
# ============================================================================

class MultiModelClassifier:
    """Train and compare multiple models on bioinformatic features."""
    
    def __init__(self, random_state=42):
        self.random_state = random_state
        self.models = {}
        self.class_mapping = {}
        
    def load_and_prepare_data(self, excel_path: str) -> Tuple[np.ndarray, np.ndarray, pd.DataFrame]:
        """Load and prepare data."""
        print("[1/3] Loading data...")
        df = pd.read_excel(excel_path)
        print(f"  Loaded: {len(df)} samples")
        
        df.columns = df.columns.str.lower().str.strip()
        df['sequence'] = df['sequence'].str.upper().str.strip()
        df['category'] = df['category'].str.lower().str.strip()
        df['lipidation'] = df['lipidation'].fillna('none').str.lower().str.strip()
        
        valid_categories = ['soluble', 'llps', 'aggregate']
        df = df[df['category'].isin(valid_categories)].copy()
        
        label_mapping = {'soluble': 0, 'llps': 1, 'aggregate': 2}
        df['label'] = df['category'].map(label_mapping)
        self.class_mapping = {v: k for k, v in label_mapping.items()}
        
        print(f"  Categories: {df['category'].value_counts().to_dict()}")
        
        sequences = df['sequence'].values
        labels = df['label'].values
        
        return sequences, labels, df
    
    def train_models(self, X: np.ndarray, y: np.ndarray) -> Dict:
        """Train multiple models."""
        print("\n[3/3] Training multiple models...")
        
        train_idx, test_idx = train_test_split(
            np.arange(len(y)), 
            test_size=0.2, 
            stratify=y, 
            random_state=self.random_state
        )
        
        X_train = X[train_idx]
        X_test = X[test_idx]
        y_train = y[train_idx]
        y_test = y[test_idx]
        
        print(f"  Train: {len(train_idx)}, Test: {len(test_idx)}")
        print(f"  Train distribution: {Counter(y_train)}")
        print(f"  Test distribution: {Counter(y_test)}")
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        print(f"  Feature shape: {X_train_scaled.shape}")
        print(f"  Scaled ranges: [{X_train_scaled.min():.3f}, {X_train_scaled.max():.3f}]")
        
        try:
            smote = SMOTE(random_state=self.random_state, k_neighbors=3)
            X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)
            print(f"  After SMOTE: {Counter(y_train_resampled)}")
        except Exception as e:
            print(f"  ⚠️  SMOTE failed: {e}. Using original data.")
            X_train_resampled, y_train_resampled = X_train_scaled, y_train
        
        results_dict = {}
        
        # ========== LOGISTIC REGRESSION ==========
        print("\n  • Training Logistic Regression...")
        try:
            param_grid_lr = {'C': np.logspace(-3, 3, 10)}
            clf_lr = LogisticRegression(penalty='l2', solver='lbfgs', 
                                       class_weight='balanced', max_iter=5000, 
                                       random_state=self.random_state)
            grid_lr = GridSearchCV(clf_lr, param_grid_lr, cv=3, scoring='f1_macro', n_jobs=-1)
            grid_lr.fit(X_train_resampled, y_train_resampled)
            best_lr = grid_lr.best_estimator_
            self.models['LogisticRegression'] = best_lr
            
            y_pred_lr = best_lr.predict(X_test_scaled)
            y_proba_lr = best_lr.predict_proba(X_test_scaled)
            
            results_dict['LogisticRegression'] = {
                'model': best_lr,
                'predictions': y_pred_lr,
                'probabilities': y_proba_lr,
            }
            print(f"    Best C: {grid_lr.best_params_['C']:.4f}")
        except Exception as e:
            print(f"    ❌ Failed: {e}")
        
        # ========== RANDOM FOREST ==========
        print("\n  • Training Random Forest...")
        try:
            clf_rf = RandomForestClassifier(n_estimators=100, random_state=self.random_state,
                                           class_weight='balanced', n_jobs=-1, max_depth=15)
            clf_rf.fit(X_train_resampled, y_train_resampled)
            self.models['RandomForest'] = clf_rf
            
            y_pred_rf = clf_rf.predict(X_test_scaled)
            y_proba_rf = clf_rf.predict_proba(X_test_scaled)
            
            results_dict['RandomForest'] = {
                'model': clf_rf,
                'predictions': y_pred_rf,
                'probabilities': y_proba_rf,
            }
            print(f"    Fitted successfully")
        except Exception as e:
            print(f"    ❌ Failed: {e}")
        
        # ========== GRADIENT BOOSTING ==========
        print("\n  • Training Gradient Boosting...")
        try:
            clf_gb = GradientBoostingClassifier(n_estimators=100, random_state=self.random_state,
                                              max_depth=5, learning_rate=0.1)
            clf_gb.fit(X_train_resampled, y_train_resampled)
            self.models['GradientBoosting'] = clf_gb
            
            y_pred_gb = clf_gb.predict(X_test_scaled)
            y_proba_gb = clf_gb.predict_proba(X_test_scaled)
            
            results_dict['GradientBoosting'] = {
                'model': clf_gb,
                'predictions': y_pred_gb,
                'probabilities': y_proba_gb,
            }
            print(f"    Fitted successfully")
        except Exception as e:
            print(f"    ❌ Failed: {e}")
        
        # ========== SUPPORT VECTOR MACHINE ==========
        print("\n  • Training Support Vector Machine...")
        try:
            clf_svm = SVC(kernel='rbf', C=1.0, probability=True, 
                         class_weight='balanced', random_state=self.random_state)
            clf_svm.fit(X_train_resampled, y_train_resampled)
            self.models['SVM'] = clf_svm
            
            y_pred_svm = clf_svm.predict(X_test_scaled)
            y_proba_svm = clf_svm.predict_proba(X_test_scaled)
            
            results_dict['SVM'] = {
                'model': clf_svm,
                'predictions': y_pred_svm,
                'probabilities': y_proba_svm,
            }
            print(f"    Fitted successfully")
        except Exception as e:
            print(f"    ❌ Failed: {e}")
        
        # ========== LIGHTGBM ==========
        print("\n  • Training LightGBM...")
        try:
            clf_lgbm = LGBMClassifier(n_estimators=200, num_leaves=31, 
                                     learning_rate=0.05, class_weight='balanced',
                                     random_state=self.random_state, n_jobs=-1, 
                                     verbose=-1)
            clf_lgbm.fit(X_train_resampled, y_train_resampled)
            self.models['LightGBM'] = clf_lgbm
            
            y_pred_lgbm = clf_lgbm.predict(X_test_scaled)
            y_proba_lgbm = clf_lgbm.predict_proba(X_test_scaled)
            
            results_dict['LightGBM'] = {
                'model': clf_lgbm,
                'predictions': y_pred_lgbm,
                'probabilities': y_proba_lgbm,
            }
            print(f"    Fitted successfully")
        except Exception as e:
            print(f"    ❌ Failed: {e}")
        
        # ========== XGBOOST ==========
        if HAS_XGB:
            print("\n  • Training XGBoost...")
            try:
                clf_xgb = xgb.XGBClassifier(n_estimators=100, random_state=self.random_state,
                                           scale_pos_weight=1, verbose=0, n_jobs=-1)
                clf_xgb.fit(X_train_resampled, y_train_resampled)
                self.models['XGBoost'] = clf_xgb
                
                y_pred_xgb = clf_xgb.predict(X_test_scaled)
                y_proba_xgb = clf_xgb.predict_proba(X_test_scaled)
                
                results_dict['XGBoost'] = {
                    'model': clf_xgb,
                    'predictions': y_pred_xgb,
                    'probabilities': y_proba_xgb,
                }
                print(f"    Fitted successfully")
            except Exception as e:
                print(f"    ❌ Failed: {e}")
        
        return results_dict, X_test_scaled, y_test, scaler


# ============================================================================
# PART 5: METRICS COMPUTATION
# ============================================================================

def compute_metrics(y_true, y_pred, y_proba, model_name, class_names):
    """Compute comprehensive metrics for a model."""
    
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    
    try:
        auc_score = roc_auc_score(y_true, y_proba, multi_class='ovr', average='weighted')
    except Exception:
        auc_score = 0.0
    
    return {
        'Model': model_name,
        'Accuracy': acc,
        'Balanced_Accuracy': bal_acc,
        'Precision': prec,
        'Recall': rec,
        'F1_Score': f1,
        'MCC': mcc,
        'AUROC': auc_score,
    }


def plot_model_comparison(metrics_df, output_dir):
    """Plot model comparison."""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12), dpi=300)
    
    # Accuracy metrics
    ax = axes[0, 0]
    x = np.arange(len(metrics_df))
    width = 0.15
    
    ax.bar(x - width*1.5, metrics_df['Accuracy'], width, label='Accuracy', alpha=0.8, edgecolor='black')
    ax.bar(x - width*0.5, metrics_df['Balanced_Accuracy'], width, label='Balanced Accuracy', alpha=0.8, edgecolor='black')
    ax.bar(x + width*0.5, metrics_df['Precision'], width, label='Precision', alpha=0.8, edgecolor='black')
    ax.bar(x + width*1.5, metrics_df['Recall'], width, label='Recall', alpha=0.8, edgecolor='black')
    
    ax.set_ylabel('Score', fontweight='bold', fontsize=12)
    ax.set_title('Accuracy Metrics Comparison', fontweight='bold', fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_df['Model'], rotation=45, ha='right')
    ax.legend(fontsize=10)
    ax.set_ylim([0, 1.1])
    ax.grid(axis='y', alpha=0.3)
    
    # F1 and MCC
    ax = axes[0, 1]
    ax.bar(x - width/2, metrics_df['F1_Score'], width, label='F1 Score', alpha=0.8, edgecolor='black', color='#2ecc71')
    ax.bar(x + width/2, metrics_df['MCC'], width, label='MCC', alpha=0.8, edgecolor='black', color='#f39c12')
    
    ax.set_ylabel('Score', fontweight='bold', fontsize=12)
    ax.set_title('F1 Score and MCC Comparison', fontweight='bold', fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_df['Model'], rotation=45, ha='right')
    ax.legend(fontsize=10)
    ax.set_ylim([0, 1.1])
    ax.grid(axis='y', alpha=0.3)
    
    # AUROC
    ax = axes[1, 0]
    colors_auc = ['#3498db' if val > 0.7 else '#e74c3c' for val in metrics_df['AUROC']]
    bars = ax.bar(metrics_df['Model'], metrics_df['AUROC'], alpha=0.8, edgecolor='black', linewidth=2, color=colors_auc)
    
    for bar, val in zip(bars, metrics_df['AUROC']):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, height + 0.02, f'{val:.3f}',
               ha='center', va='bottom', fontweight='bold', fontsize=11)
    
    ax.set_ylabel('AUROC', fontweight='bold', fontsize=12)
    ax.set_title('AUROC Comparison', fontweight='bold', fontsize=14)
    ax.set_ylim([0, 1.1])
    ax.grid(axis='y', alpha=0.3)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # Best model summary
    ax = axes[1, 1]
    ax.axis('off')
    
    best_idx = metrics_df['Accuracy'].idxmax()
    best_model = metrics_df.iloc[best_idx]
    
    summary_text = f"""
    BEST MODEL SUMMARY
    {'='*40}
    
    Model: {best_model['Model']}
    
    Accuracy:           {best_model['Accuracy']:.4f}
    Balanced Accuracy:  {best_model['Balanced_Accuracy']:.4f}
    Precision:          {best_model['Precision']:.4f}
    Recall:             {best_model['Recall']:.4f}
    F1 Score:           {best_model['F1_Score']:.4f}
    MCC:                {best_model['MCC']:.4f}
    AUROC:              {best_model['AUROC']:.4f}
    """
    
    ax.text(0.05, 0.95, summary_text, transform=ax.transAxes, fontsize=11,
           verticalalignment='top', fontfamily='monospace',
           bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8, pad=1))
    
    plt.suptitle('Multi-Model Performance Comparison', fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig(output_dir / '10_model_comparison.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Model comparison plot: 10_model_comparison.png")


# ============================================================================
# PART 6: EXPERIMENTAL VALIDATION WITH ROC CURVES
# ============================================================================

def validate_on_experimental(exp_file, best_model, best_model_name, scaler, extractor, 
                            output_dir, class_mapping):
    """Validate on experimental dataset with ROC curves."""
    print(f"\nValidating on experimental data using {best_model_name}...")
    
    try:
        df_exp = pd.read_excel(exp_file)
    except Exception as e:
        print(f"❌ Failed to load experimental file: {e}")
        return None
    
    df_exp.columns = df_exp.columns.str.lower().str.strip()
    df_exp['sequence'] = df_exp['sequence'].str.upper().str.strip()
    df_exp['category'] = df_exp['category'].str.lower().str.strip()
    df_exp['lipidation'] = df_exp['lipidation'].fillna('none').str.lower().str.strip()
    
    valid_categories = ['soluble', 'llps', 'aggregate']
    df_exp = df_exp[df_exp['category'].isin(valid_categories)].copy()
    
    if len(df_exp) == 0:
        print("❌ No valid experimental samples!")
        return None
    
    print(f"  Experimental samples: {len(df_exp)}")
    print(f"  Categories: {df_exp['category'].value_counts().to_dict()}")
    
    sequences = df_exp['sequence'].values
    lipidations = df_exp['lipidation'].values
    
    label_map = {'soluble': 0, 'llps': 1, 'aggregate': 2}
    y_true = df_exp['category'].map(label_map).values
    
    print(f"\n  Extracting bioinformatic features...")
    X_exp, _ = extractor.extract_all_features(sequences, lipidations)
    X_exp = np.nan_to_num(X_exp, nan=0.0, posinf=0.0, neginf=0.0)
    
    expected_features = scaler.n_features_in_
    if X_exp.shape[1] != expected_features:
        print(f"❌ Feature dimension mismatch!")
        return None
    
    X_exp_scaled = scaler.transform(X_exp)
    
    y_pred = best_model.predict(X_exp_scaled)
    y_proba = best_model.predict_proba(X_exp_scaled)
    
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    
    try:
        auc_score = roc_auc_score(y_true, y_proba, multi_class='ovr', average='weighted')
    except Exception:
        auc_score = 0.0
    
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    
    # Confusion matrix plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), dpi=300)
    
    inv_label_map = {v: k for k, v in label_map.items()}
    class_names = [inv_label_map[i].upper() for i in range(3)]
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
                xticklabels=class_names, yticklabels=class_names,
                annot_kws={"size": 12}, linewidths=2)
    axes[0].set_xlabel('Predicted', fontweight='bold', fontsize=12)
    axes[0].set_ylabel('True', fontweight='bold', fontsize=12)
    axes[0].set_title('Confusion Matrix - External Validation (Counts)', fontweight='bold', fontsize=13)
    
    cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues', cbar=True, ax=axes[1],
                xticklabels=class_names, yticklabels=class_names,
                annot_kws={"size": 12}, cbar_kws={"label": "Percentage (%)"}, linewidths=2)
    axes[1].set_xlabel('Predicted', fontweight='bold', fontsize=12)
    axes[1].set_ylabel('True', fontweight='bold', fontsize=12)
    axes[1].set_title('Confusion Matrix - External Validation (%)', fontweight='bold', fontsize=13)
    
    plt.tight_layout()
    plt.savefig(output_dir / '06_confusion_matrix_external_validation.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    # ROC curves
    roc_viz = ROCVisualization()
    
    fpr_ext, tpr_ext, auc_ext, auc_per_class_ext = roc_viz.plot_macro_roc_curve(
        y_true, y_proba, class_names,
        title='Macro-Averaged ROC Curve - External Validation',
        output_path=output_dir / '07_roc_macro_external_validation.png'
    )
    
    roc_viz.plot_per_class_roc_curves(
        y_true, y_proba, class_names,
        title='Per-Class ROC Curves - External Validation',
        output_path=output_dir / '08_roc_per_class_external_validation.png'
    )
    
    # Save predictions
    df_exp['predicted_category'] = [inv_label_map[p] for p in y_pred]
    df_exp['correct'] = df_exp['category'] == df_exp['predicted_category']
    df_exp['prob_soluble'] = y_proba[:, 0]
    df_exp['prob_llps'] = y_proba[:, 1]
    df_exp['prob_aggregate'] = y_proba[:, 2]
    df_exp['confidence'] = y_proba.max(axis=1)
    
    df_exp.to_csv(output_dir / 'experimental_predictions.csv', index=False)
    
    print(f"\n{'='*60}")
    print(f"EXTERNAL VALIDATION RESULTS ({best_model_name})")
    print(f"{'='*60}")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  Balanced Accuracy: {bal_acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall: {rec:.4f}")
    print(f"  F1 (weighted): {f1:.4f}")
    print(f"  MCC: {mcc:.4f}")
    print(f"  AUROC (weighted): {auc_score:.4f}")
    
    print(f"\n  Per-class ROC-AUC:")
    for class_name, auc_val in auc_per_class_ext.items():
        print(f"    {class_name.upper()}: {auc_val:.4f}")
    
    print(f"\n  Per-class metrics:")
    for class_idx in range(3):
        class_name = inv_label_map[class_idx].upper()
        mask = y_true == class_idx
        if mask.sum() > 0:
            class_acc = (y_pred[mask] == class_idx).sum() / mask.sum()
            
            y_true_binary = (y_true == class_idx).astype(int)
            y_pred_binary = (y_pred == class_idx).astype(int)
            
            if y_true_binary.sum() > 0 or y_pred_binary.sum() > 0:
                class_prec = precision_score(y_true_binary, y_pred_binary, 
                                            zero_division=0, average='binary')
                class_rec = recall_score(y_true_binary, y_pred_binary, 
                                        zero_division=0, average='binary')
                class_f1 = f1_score(y_true_binary, y_pred_binary, 
                                   zero_division=0, average='binary')
                print(f"    {class_name}: Acc={class_acc:.3f}, Prec={class_prec:.3f}, Rec={class_rec:.3f}, F1={class_f1:.3f} (n={mask.sum()})")
    
    print(f"{'='*60}")
    
    return {
        'accuracy': acc,
        'balanced_accuracy': bal_acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'mcc': mcc,
        'auroc': auc_score,
        'auroc_per_class': auc_per_class_ext,
        'confusion_matrix': cm,
        'predictions_df': df_exp,
        'fpr': fpr_ext,
        'tpr': tpr_ext,
        'auc': auc_ext,
    }


# ============================================================================
# PART 7: FEATURE IMPORTANCE ANALYSIS
# ============================================================================

def analyze_feature_importance(model, feature_names, output_dir):
    """Analyze and visualize feature importance."""
    print("\nAnalyzing feature importance...")
    
    if not hasattr(model, 'feature_importances_'):
        print("  Model doesn't have feature_importances_ attribute")
        return None
    
    importances = model.feature_importances_
    n_features_actual = len(importances)
    n_features_named = len(feature_names)
    
    if n_features_actual != n_features_named:
        print(f"  ⚠️  Feature count mismatch: {n_features_actual} vs {n_features_named}")
        if n_features_actual > n_features_named:
            extra = n_features_actual - n_features_named
            feature_names = list(feature_names) + [f'feature_{n_features_named + i}' for i in range(extra)]
        else:
            feature_names = feature_names[:n_features_actual]
    
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances,
    }).sort_values('Importance', ascending=False)
    
    def classify_feature(feat_name):
        if 'complexity' in feat_name:
            return 'Complexity'
        elif 'ss_' in feat_name:
            return 'Secondary Structure'
        elif 'hydro' in feat_name:
            return 'Hydrophobic'
        elif 'charge' in feat_name:
            return 'Charge'
        elif 'aromatic' in feat_name:
            return 'Aromatic'
        elif 'frac_' in feat_name or 'pos' in feat_name:
            return 'Composition'
        elif 'lipid' in feat_name:
            return 'Lipidation'
        else:
            return 'Other'
    
    importance_df['Category'] = importance_df['Feature'].apply(classify_feature)
    
    print(f"  Feature categories:")
    for cat in importance_df['Category'].unique():
        count = (importance_df['Category'] == cat).sum()
        importance = importance_df[importance_df['Category'] == cat]['Importance'].sum()
        print(f"    {cat}: {count} features, {importance:.4f} importance")
    
    fig, ax = plt.subplots(figsize=(12, 14), dpi=300)
    top30 = importance_df.head(30).iloc[::-1]
    
    cat_colors = {
        'Complexity': '#f39c12',
        'Secondary Structure': '#3498db',
        'Hydrophobic': '#2ecc71',
        'Charge': '#9b59b6',
        'Aromatic': '#1abc9c',
        'Composition': '#34495e',
        'Lipidation': '#e67e22',
        'Other': '#95a5a6',
    }
    
    colors_imp = [cat_colors.get(cat, '#95a5a6') for cat in top30['Category']]
    
    ax.barh(range(len(top30)), top30['Importance'].values, color=colors_imp, 
            alpha=0.7, edgecolor='black', linewidth=0.5)
    ax.set_yticks(range(len(top30)))
    ax.set_yticklabels(top30['Feature'].values, fontsize=9)
    ax.set_xlabel('Feature Importance', fontweight='bold')
    ax.set_title('Top 30 Bioinformatic Features', fontweight='bold', fontsize=14)
    ax.grid(axis='x', alpha=0.3)
    
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=color, label=cat) 
                      for cat, color in cat_colors.items() 
                      if cat in top30['Category'].unique()]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
    plt.tight_layout()
    plt.savefig(output_dir / '11_feature_importance_top30.png', dpi=300, bbox_inches='tight')
    plt.close()

    fig, ax = plt.subplots(figsize=(12, 6), dpi=300)

    cat_importance = importance_df.groupby('Category')['Importance'].sum().sort_values(ascending=False)
    cat_counts = importance_df.groupby('Category').size()

    colors_cat = [cat_colors.get(cat, '#95a5a6') for cat in cat_importance.index]

    bars = ax.bar(range(len(cat_importance)), cat_importance.values, color=colors_cat,
                alpha=0.7, edgecolor='black', linewidth=2)

    ax.set_xticks(range(len(cat_importance)))
    ax.set_xticklabels(cat_importance.index, rotation=45, ha='right')
    ax.set_ylabel('Total Feature Importance', fontweight='bold', fontsize=12)
    ax.set_title('Feature Importance by Category', fontweight='bold', fontsize=14)
    ax.grid(axis='y', alpha=0.3)

    for i, (bar, val) in enumerate(zip(bars, cat_importance.values)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, height + 0.005,
            f'{val:.3f}\n(n={cat_counts[cat_importance.index[i]]})',
            ha='center', va='bottom', fontweight='bold', fontsize=9)

    plt.tight_layout()
    plt.savefig(output_dir / '12_feature_importance_by_category.png', dpi=300, bbox_inches='tight')
    plt.close()

    importance_df.to_csv(output_dir / 'feature_importance_detailed.csv', index=False)

    return importance_df


# ============================================================================
# PART 8: VALIDATION CHECK
# ============================================================================

def quick_validation_check(config):
    """Quick validation before running pipeline."""
    print("\n" + "="*80)
    print("QUICK VALIDATION CHECKS")
    print("="*80)
    
    print("\n[1/4] Checking file paths...")
    training_file = Path(config['training_file'])
    experimental_file = Path(config['experimental_file'])
    
    if not training_file.exists():
        print(f"❌ Training file not found: {training_file}")
        return False
    print(f"✓ Training file found")
    
    if not experimental_file.exists():
        print(f"❌ Experimental file not found: {experimental_file}")
        return False
    print(f"✓ Experimental file found")
    
    print("\n[2/4] Checking output directory...")
    output_dir = Path(config['output_dir'])
    try:
        output_dir.mkdir(exist_ok=True, parents=True)
        test_file = output_dir / '.test_write'
        test_file.write_text('test')
        test_file.unlink()
        print(f"✓ Output directory writable")
    except Exception as e:
        print(f"❌ Cannot write to output directory: {e}")
        return False
    
    print("\n[3/4] Quick data loading check...")
    try:
        df_train = pd.read_excel(training_file, nrows=10)
        print(f"✓ Training file readable: {df_train.shape}")
        
        df_exp = pd.read_excel(experimental_file, nrows=10)
        print(f"✓ Experimental file readable: {df_exp.shape}")
    except Exception as e:
        print(f"❌ Error reading files: {e}")
        return False
    
    print("\n[4/4] Checking required columns...")
    required_cols = ['sequence', 'category', 'lipidation']
    
    for col in required_cols:
        if col not in [c.lower() for c in df_train.columns]:
            print(f"❌ Training file missing '{col}' column")
            return False
    print(f"✓ Training file has required columns")
    
    for col in required_cols:
        if col not in [c.lower() for c in df_exp.columns]:
            print(f"❌ Experimental file missing '{col}' column")
            return False
    print(f"✓ Experimental file has required columns")
    
    print("\n" + "="*80)
    print("✅ ALL VALIDATION CHECKS PASSED")
    print("="*80)
    return True


# ============================================================================
# PART 9: MAIN WORKFLOW
# ============================================================================

def main():
    """Complete bioinformatic workflow with multi-model comparison."""
    
    if not quick_validation_check(CONFIG):
        print("\n❌ Validation failed. Exiting.")
        return
    
    print("\n" + "="*80)
    print("LLPS PEPTIDE CLASSIFICATION - BIOINFORMATIC ANALYSIS")
    print("Multi-Model Comparison with Comprehensive ROC Analysis")
    print("="*80)
    
    output_dir = Path(CONFIG['output_dir'])
    output_dir.mkdir(exist_ok=True, parents=True)
    
    # ========== LOAD DATA ==========
    classifier = MultiModelClassifier(random_state=CONFIG['random_state'])
    sequences, labels, metadata = classifier.load_and_prepare_data(CONFIG['training_file'])
    
    max_seq_len = max(len(s) for s in sequences)
    print(f"  Max sequence length: {max_seq_len}")
    
    # ========== EXTRACT FEATURES ==========
    print("\n[2/3] Extracting bioinformatic features...")
    print("  This may take a few minutes...")
    
    extractor = BioinformaticFeatureExtractor(CONFIG)
    extractor.max_len = max_seq_len
    
    lipidations = metadata['lipidation'].values
    X_bio, names_bio = extractor.extract_all_features(sequences, lipidations)
    
    X_bio = np.nan_to_num(X_bio, nan=0.0, posinf=0.0, neginf=0.0)
    
    print(f"\nFeature Summary:")
    print(f"  Total Bioinformatic Features: {X_bio.shape[1]}")
    print(f"  Samples: {X_bio.shape[0]}")
    
    print(f"\nFeature Categories:")
    print(f"  Position encoding: {sum(1 for n in names_bio if 'pos' in n)}")
    print(f"  AA composition: {sum(1 for n in names_bio if 'frac_' in n)}")
    print(f"  Charge properties: {sum(1 for n in names_bio if 'charge' in n)}")
    print(f"  Hydrophobic moment: {sum(1 for n in names_bio if 'hydro' in n)}")
    print(f"  Aromatic geometry: {sum(1 for n in names_bio if 'aromatic' in n)}")
    print(f"  Lipidation: {sum(1 for n in names_bio if 'lipid' in n)}")
    print(f"  Low complexity: {sum(1 for n in names_bio if 'complexity' in n)}")
    print(f"  Secondary structure: {sum(1 for n in names_bio if 'ss_' in n)}")
    
    # ========== TRAIN MULTIPLE MODELS ==========
    results_dict, X_test_scaled, y_test, best_scaler = classifier.train_models(X_bio, labels)
    
    # ========== COMPUTE METRICS FOR ALL MODELS ==========
    print("\n" + "="*80)
    print("COMPUTING METRICS FOR ALL MODELS")
    print("="*80)
    
    class_names = ['SOLUBLE', 'LLPS', 'AGGREGATE']
    metrics_list = []
    
    for model_name, results in results_dict.items():
        print(f"\n{model_name}:")
        y_pred = results['predictions']
        y_proba = results['probabilities']
        
        metrics = compute_metrics(y_test, y_pred, y_proba, model_name, class_names)
        metrics_list.append(metrics)
        
        print(f"  Accuracy: {metrics['Accuracy']:.4f}")
        print(f"  Balanced Accuracy: {metrics['Balanced_Accuracy']:.4f}")
        print(f"  Precision: {metrics['Precision']:.4f}")
        print(f"  Recall: {metrics['Recall']:.4f}")
        print(f"  F1 Score: {metrics['F1_Score']:.4f}")
        print(f"  MCC: {metrics['MCC']:.4f}")
        print(f"  AUROC: {metrics['AUROC']:.4f}")
    
    metrics_df = pd.DataFrame(metrics_list).sort_values('Accuracy', ascending=False)
    
    # Save metrics
    metrics_df.to_csv(output_dir / 'model_metrics_comparison.csv', index=False)
    print(f"\n✓ Metrics saved: model_metrics_comparison.csv")
    
    # ========== PLOT MODEL COMPARISON ==========
    print("\n" + "="*80)
    print("PLOTTING MODEL COMPARISON")
    print("="*80)
    plot_model_comparison(metrics_df, output_dir)
    
    # ========== SELECT BEST MODEL ==========
    best_idx = metrics_df['Accuracy'].idxmax()
    best_model_name = metrics_df.iloc[best_idx]['Model']
    best_model_obj = results_dict[best_model_name]['model']
    
    print(f"\n🏆 BEST MODEL: {best_model_name}")
    print(f"   Accuracy: {metrics_df.iloc[best_idx]['Accuracy']:.4f}")
    
    # ========== GENERATE INTERNAL TEST CONFUSION MATRIX & ROC ==========
    print("\n" + "="*80)
    print("INTERNAL TEST SET ANALYSIS")
    print("="*80)
    
    y_pred_test = results_dict[best_model_name]['predictions']
    y_proba_test = results_dict[best_model_name]['probabilities']
    
    # Confusion matrix
    print("\n  Generating confusion matrix...")
    cm_test = confusion_matrix(y_test, y_pred_test, labels=[0, 1, 2])
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), dpi=300)
    
    sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
                xticklabels=class_names, yticklabels=class_names,
                annot_kws={"size": 12}, linewidths=2)
    axes[0].set_xlabel('Predicted', fontweight='bold', fontsize=12)
    axes[0].set_ylabel('True', fontweight='bold', fontsize=12)
    axes[0].set_title('Confusion Matrix - Internal Test (Counts)', fontweight='bold', fontsize=13)
    
    cm_pct = cm_test.astype('float') / cm_test.sum(axis=1)[:, np.newaxis] * 100
    sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues', cbar=True, ax=axes[1],
                xticklabels=class_names, yticklabels=class_names,
                annot_kws={"size": 12}, cbar_kws={"label": "Percentage (%)"}, linewidths=2)
    axes[1].set_xlabel('Predicted', fontweight='bold', fontsize=12)
    axes[1].set_ylabel('True', fontweight='bold', fontsize=12)
    axes[1].set_title('Confusion Matrix - Internal Test (%)', fontweight='bold', fontsize=13)
    
    plt.tight_layout()
    plt.savefig(output_dir / '02_confusion_matrix_internal_test.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Confusion matrix (internal test): 02_confusion_matrix_internal_test.png")
    
    # ROC curves
    roc_viz = ROCVisualization()
    
    print("\n  Generating macro-averaged ROC curve...")
    fpr_int, tpr_int, auc_int, auc_per_class_int = roc_viz.plot_macro_roc_curve(
        y_test, y_proba_test, class_names,
        title='Macro-Averaged ROC Curve - Internal Test',
        output_path=output_dir / '03_roc_macro_internal_test.png'
    )
    
    print("\n  Generating per-class ROC curves...")
    roc_viz.plot_per_class_roc_curves(
        y_test, y_proba_test, class_names,
        title='Per-Class ROC Curves - Internal Test',
        output_path=output_dir / '04_roc_per_class_internal_test.png'
    )
    
    # Print internal test metrics
    print(f"\n{'='*60}")
    print(f"INTERNAL TEST SET RESULTS ({best_model_name})")
    print(f"{'='*60}")
    
    acc_int = accuracy_score(y_test, y_pred_test)
    bal_acc_int = balanced_accuracy_score(y_test, y_pred_test)
    prec_int = precision_score(y_test, y_pred_test, average='weighted', zero_division=0)
    rec_int = recall_score(y_test, y_pred_test, average='weighted', zero_division=0)
    f1_int = f1_score(y_test, y_pred_test, average='weighted', zero_division=0)
    mcc_int = matthews_corrcoef(y_test, y_pred_test)
    
    print(f"  Accuracy: {acc_int:.4f}")
    print(f"  Balanced Accuracy: {bal_acc_int:.4f}")
    print(f"  Precision: {prec_int:.4f}")
    print(f"  Recall: {rec_int:.4f}")
    print(f"  F1 (weighted): {f1_int:.4f}")
    print(f"  MCC: {mcc_int:.4f}")
    print(f"  AUROC (macro): {auc_int:.4f}")
    
    print(f"\n  Per-class ROC-AUC:")
    for class_name, auc_val in auc_per_class_int.items():
        print(f"    {class_name.upper()}: {auc_val:.4f}")
    
    print(f"\n  Per-class metrics:")
    for class_idx in range(3):
        class_name = class_names[class_idx]
        mask = y_test == class_idx
        if mask.sum() > 0:
            class_acc = (y_pred_test[mask] == class_idx).sum() / mask.sum()
            
            y_test_binary = (y_test == class_idx).astype(int)
            y_pred_binary = (y_pred_test == class_idx).astype(int)
            
            if y_test_binary.sum() > 0 or y_pred_binary.sum() > 0:
                class_prec = precision_score(y_test_binary, y_pred_binary, 
                                            zero_division=0, average='binary')
                class_rec = recall_score(y_test_binary, y_pred_binary, 
                                        zero_division=0, average='binary')
                class_f1 = f1_score(y_test_binary, y_pred_binary, 
                                   zero_division=0, average='binary')
                print(f"    {class_name}: Acc={class_acc:.3f}, Prec={class_prec:.3f}, Rec={class_rec:.3f}, F1={class_f1:.3f} (n={mask.sum()})")
    
    print(f"{'='*60}")
    
    # ========== EXPERIMENTAL VALIDATION ==========
    print("\n" + "="*80)
    print("EXTERNAL VALIDATION")
    print("="*80)
    
    exp_results = validate_on_experimental(
        CONFIG['experimental_file'],
        best_model_obj,
        best_model_name,
        best_scaler,
        extractor,
        output_dir,
        classifier.class_mapping
    )
    
    if exp_results is None:
        print("❌ Experimental validation failed")
        exp_results = {'balanced_accuracy': 0.0, 'auroc': 0.0}
    
    # ========== ROC COMPARISON ==========
    if exp_results and 'fpr' in exp_results:
        print("\n" + "="*80)
        print("ROC CURVE COMPARISON")
        print("="*80)
        
        roc_viz.plot_roc_comparison(
            {
                'Internal Test': (fpr_int, tpr_int, auc_int),
                'External Validation': (exp_results['fpr'], exp_results['tpr'], exp_results['auc'])
            },
            output_path=output_dir / '09_roc_comparison_internal_vs_external.png'
        )
    
    # ========== FEATURE IMPORTANCE ==========
    print("\n" + "="*80)
    print("FEATURE IMPORTANCE ANALYSIS")
    print("="*80)
    
    importance_df = analyze_feature_importance(best_model_obj, names_bio, output_dir)
    
    if importance_df is not None:
        print(f"\nTop 10 Most Important Features:")
        print(importance_df.head(10).to_string(index=False))
    
    # ========== SAVE MODELS ==========
    print("\n" + "="*80)
    print("SAVING MODELS")
    print("="*80)
    
    joblib.dump(best_model_obj, output_dir / f'best_model_{best_model_name}.pkl')
    joblib.dump(best_scaler, output_dir / 'scaler.pkl')
    joblib.dump(names_bio, output_dir / 'feature_names.pkl')
    
    print(f"\n✓ Best model: best_model_{best_model_name}.pkl")
    print(f"✓ Scaler: scaler.pkl")
    print(f"✓ Feature names: feature_names.pkl")
    
    # ========== FINAL SUMMARY ==========
    print("\n" + "="*80)
    print("FINAL SUMMARY")
    print("="*80)
    
    summary = {
        'Training_Samples': len(sequences),
        'N_Total_Bioinformatic_Features': X_bio.shape[1],
        'Best_Model': best_model_name,
        'Internal_Test_Accuracy': acc_int,
        'Internal_Test_Balanced_Accuracy': bal_acc_int,
        'Internal_Test_Precision': prec_int,
        'Internal_Test_Recall': rec_int,
        'Internal_Test_F1_Score': f1_int,
        'Internal_Test_MCC': mcc_int,
        'Internal_Test_AUROC': auc_int,
        'External_Test_Accuracy': exp_results.get('accuracy', 0.0),
        'External_Test_Balanced_Accuracy': exp_results.get('balanced_accuracy', 0.0),
        'External_Test_Precision': exp_results.get('precision', 0.0),
        'External_Test_Recall': exp_results.get('recall', 0.0),
        'External_Test_F1_Score': exp_results.get('f1', 0.0),
        'External_Test_MCC': exp_results.get('mcc', 0.0),
        'External_Test_AUROC': exp_results.get('auroc', 0.0),
        'Number_of_Models_Trained': len(results_dict),
        'Models_Compared': ', '.join(metrics_df['Model'].tolist()),
    }
    
    summary_df = pd.DataFrame([summary])
    summary_df.to_csv(output_dir / 'final_summary.csv', index=False)
    
    print("\n📊 MODEL COMPARISON TABLE:")
    print(metrics_df.to_string(index=False))
    
    print("\n📊 INTERNAL TEST SET:")
    print(f"  Best Model: {best_model_name}")
    print(f"  Accuracy: {acc_int:.4f}")
    print(f"  Balanced Accuracy: {bal_acc_int:.4f}")
    print(f"  Precision: {prec_int:.4f}")
    print(f"  Recall: {rec_int:.4f}")
    print(f"  F1 Score: {f1_int:.4f}")
    print(f"  MCC: {mcc_int:.4f}")
    print(f"  Macro AUROC: {auc_int:.4f}")
    
    print("\n📊 EXTERNAL VALIDATION SET:")
    print(f"  Accuracy: {exp_results.get('accuracy', 0.0):.4f}")
    print(f"  Balanced Accuracy: {exp_results.get('balanced_accuracy', 0.0):.4f}")
    print(f"  Precision: {exp_results.get('precision', 0.0):.4f}")
    print(f"  Recall: {exp_results.get('recall', 0.0):.4f}")
    print(f"  F1 Score: {exp_results.get('f1', 0.0):.4f}")
    print(f"  MCC: {exp_results.get('mcc', 0.0):.4f}")
    print(f"  Macro AUROC: {exp_results.get('auroc', 0.0):.4f}")
    
    print(f"\n📁 OUTPUT DIRECTORY: {output_dir}")
    
    print(f"\n📄 Output files:")
    for f in sorted(output_dir.glob('*')):
        if f.is_file():
            size = f.stat().st_size / 1024
            print(f"  {f.name:<50s} {size:>8.1f} KB")
    
    print("\n" + "="*80)
    print("✅ BIOINFORMATIC ANALYSIS COMPLETE")
    print("="*80)
    print("\n📈 Generated Outputs:")
    print("  ✓ 02_confusion_matrix_internal_test.png")
    print("  ✓ 03_roc_macro_internal_test.png + CSV data")
    print("  ✓ 04_roc_per_class_internal_test.png")
    print("  ✓ 06_confusion_matrix_external_validation.png")
    print("  ✓ 07_roc_macro_external_validation.png + CSV data")
    print("  ✓ 08_roc_per_class_external_validation.png")
    print("  ✓ 09_roc_comparison_internal_vs_external.png")
    print("  ✓ 10_model_comparison.png")
    print("  ✓ 11_feature_importance_top30.png")
    print("  ✓ 12_feature_importance_by_category.png")
    print("  ✓ model_metrics_comparison.csv")
    print("  ✓ final_summary.csv")
    print("  ✓ feature_importance_detailed.csv")
    print("  ✓ experimental_predictions.csv")
    print("  ✓ best_model_[ModelName].pkl")
    print("  ✓ scaler.pkl")
    print("  ✓ feature_names.pkl")
    print("="*80 + "\n")
    
    return metrics_df, exp_results, importance_df


if __name__ == "__main__":
    main()